In [ ]:
# Import required libraries for data manipulation, JSON parsing, and machine learning (CatBoost)
import pandas as pd
import numpy as np
import json
import catboost as cb
import os

# --- 1. Set Paths and Load Model ---
# Define file paths for the model, input points, and baseline data used for anomaly calculation
MODEL_PATH = '../models/wildfire_improved_model_d.cbm'
POINTS_JSON_PATH = '../data/district_points_data.json'
BASELINE_PATH = '../data/baseline_table.csv' # For calculating anomaly values

# Initialize and load the pre-trained CatBoost classifier
model = cb.CatBoostClassifier()
model.load_model(MODEL_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)

# --- 2. Read district point coordinate data ---
# Open and parse the JSON file containing coordinate points for each district
with open(POINTS_JSON_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

all_points = []
# Flatten the nested JSON structure into a list of dictionaries, assigning default values where missing
for province, districts in raw_data.items():
    for district, points in districts.items():
        for pt in points:
            # Check and fill default values if keys are missing in JSON
            pt['NAME_1'] = province
            pt['NAME_2'] = district
            pt['slope'] = pt.get('slope', 5)   # Default slope to 5 degrees if missing
            pt['elev'] = pt.get('elev', 200)  # Default elevation to 200 meters if missing
            pt['temp'] = pt.get('temp', 30)
            pt['ndvi'] = pt.get('ndvi', 0.4)
            all_points.append(pt)

# Convert the list of points into a pandas DataFrame for easier processing
df = pd.DataFrame(all_points)
print(f"Loaded {len(df)} points.")

# --- 3. Prepare Features ---
# Now df will definitely have 'slope' and 'elev' columns
# Calculate derived features required by the model
df['terrain_roughness'] = df['slope'] * np.log1p(df['elev'])
# ... (Remaining code as before) ...

# Assign constant values for testing or missing variables based on domain knowledge
df['wind_speed'] = 2.5
df['cluster_id'] = "1" # High risk area
df['landcover'] = "10" # Forest

# Align columns to match the model's 32 features exactly as they were during training
FEATURE_NAMES = [
    'ndvi', 'ndwi', 'nbr', 'blue', 'green', 'red', 'nir', 'swir1', 'swir2', 
    'temp', 'soil_moisture', 'wind_u', 'wind_v', 'elev', 'slope', 'aspect', 
    'landcover', 'month', 'NAME_1', 'NAME_2', 'veg_stress', 'fire_weather_idx', 
    'wind_speed', 'drought_proxy', 'terrain_roughness', 'hot_dry_stress', 
    'ndvi_anomaly', 'moisture_anomaly', 'temp_anomaly', 'month_sin', 'month_cos', 'cluster_id'
]

# Fill missing values: Ensure all required feature columns exist, filling missing ones with 0
for col in FEATURE_NAMES:
    if col not in df.columns:
        df[col] = 0

# Extract only the required feature columns for prediction
X = df[FEATURE_NAMES]

# --- 4. Predict risk using the actual model ---
# Generate probabilities for the positive class (class 1: wildfire risk)
print("Predicting risk probabilities...")
df['predicted_risk'] = model.predict_proba(X)[:, 1]

# --- 5. District-level Aggregation ---
# Scale risk to 0-100 (as discussed for better display purposes on the dashboard)
df['display_risk'] = (df['predicted_risk'] / 0.22) * 100
# Cap the maximum display risk at 95 to prevent 100% certainty overconfidence
df.loc[df['display_risk'] > 95, 'display_risk'] = 95

# Calculate the mean risk score for each district
district_summary = df.groupby(['NAME_1', 'NAME_2'])['display_risk'].mean().reset_index()

# Convert to JSON Format: { "Province": { "District": risk_value } }
# Restructure the DataFrame back into a nested dictionary for the frontend
final_json = {}
for _, row in district_summary.iterrows():
    p, d, r = row['NAME_1'], row['NAME_2'], row['display_risk']
    if p not in final_json:
        final_json[p] = {}
    final_json[p][d] = round(float(r), 2)

# --- 6. Save the file ---
# Export the final aggregated data as a JSON file
with open('../data/district_risk_summary.json', 'w', encoding='utf-8') as f:
    json.dump(final_json, f, ensure_ascii=False, indent=2)

print("✅ Success! '../data/district_risk_summary.json' has been created.")

Loaded 3941 points.
Predicting risk probabilities...
✅ Success! '../data/district_risk_summary.json' has been created.
